# Step 1 : Groupby - Average

In [1]:
# Necessary Libraries
import pandas as pd
import os
from datetime import datetime, time, timedelta
import math
import numpy as np

## Data Awal

### Data Reel

In [2]:
# Pipeline for Data Reel
def merge_reel_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file)
            dataframes.append(df)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [3]:
# Eksekusi Pipeline
daftar_file_reel = [
    "../Data Reel/data reel pm14 0326.xlsx",
    "../Data Reel/data reel pm14 0426.xlsx"
]

reel_pm14 = merge_reel_data(daftar_file_reel)
reel_pm14.head()

Berhasil memuat: ../Data Reel/data reel pm14 0326.xlsx | Shape: (557, 13)
Berhasil memuat: ../Data Reel/data reel pm14 0426.xlsx | Shape: (701, 13)


,Time,Tanggal,Grade,Shift,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,7.45,01.03.26,FC 12,1,16,12.36,0.89,413,190,120,32,86.6,Acc Sotiss
1,8.30,01.03.26,FC 12,1,17,12.68,0.96,306,148,90,30,86.5,Acc Sotiss
2,9.30,01.03.26,FC 12,1,18,12.50,0.99,352,150,96,32,86.6,Acc Sotiss
3,10.19,01.03.26,FC 12,1,19,12.24,0.91,365,155,104,32,86.3,Acc Sotiss
4,11.20,01.03.26,FC 12,1,20,12.34,0.98,294,140,81,32,86.6,Acc Sotiss


### Params PM

In [4]:
# Pipeline for Params PM
def load_and_standardize(file_path):
    # Deteksi ekstensi file untuk metode ekstraksi yang tepat
    _, ext = os.path.splitext(file_path)
    
    if ext.lower() in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif ext.lower() == '.csv':
        with open(file_path, encoding='utf-16') as f:
            raw_header = f.readline().strip()
            # Header bungkus kutip ganda — bersihkan manual
            header = raw_header.strip('"').replace('""', '').split(';')
            df = pd.read_csv(f, delimiter=';', names=header)
    else:
        raise ValueError(f"Format file tidak dikenali: {ext}")

    # Standarisasi Header
    df.columns = df.columns.str.strip()
    
    #if ext.lower() in ['.xlsx', '.xls']:
        #df.rename(columns={'Load KWH Refiner': 'Load KWH Tickling Refiner'}, inplace=True)

    rename_map = {
        'Yangkee  Speed': 'Yankee Speed',
        'Coating Flow': 'Flow Coating',
        'Release Flow': 'Flow Release',
        'Jet Rasio': 'Jet Wire Ratio',
        'Yangkee Temperatur' : 'Yankee Temperature',
        'Hood Dry Temperatur' : 'Hood Temperature1',
        'Hood Wet Temperatur' : 'Hood Temperature2',
        'Turbo Vakum Ampere' : 'Load Ampere Turbo Vacuum'
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    
    # Validasi Waktu sebagai Kunci Utama
    df.dropna(subset=['Time'], inplace=True) 
    df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, format='mixed', errors='coerce')
    
    return df

def build_master_pipeline(file_list):
    dataframes = []
    
    for file in file_list:
        try:
            df = load_and_standardize(file)
            dataframes.append(df)
        except FileNotFoundError:
            print(f"Peringatan: File {file} tidak ditemukan. Dilewati.")
            
    if not dataframes:
        raise ValueError("Tidak ada data yang berhasil dimuat.")

    # Penggabungan dan Pembersihan Data Redundan
    master_df = pd.concat(dataframes, ignore_index=True)
    master_df.drop_duplicates(subset=['Time'], keep='last', inplace=True)
    master_df.sort_values('Time', inplace=True)
    
    return master_df.reset_index(drop=True)

In [5]:
# Eksekusi Pipeline
file_sources = [
    '../PM Params/Maret-April PM 14.xlsx',
    '../PM Params/04052026_PM14.csv'
]
raw14 = build_master_pipeline(file_sources)
raw14.drop(['Yankee Temperature', 'Hood Temperature2', 'Hood Temperature1', 'Load Ampere Turbo Vacuum', 'Tekanan Yangkee'], axis=1, inplace=True)
raw14.info()

<class 'pandas.DataFrame'>
RangeIndex: 86876 entries, 0 to 86875
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Time               86876 non-null  datetime64[us]
 1   Yankee Speed       86333 non-null  float64       
 2   Pope Reel Speed    86333 non-null  float64       
 3   Yankee Pressure    86333 non-null  float64       
 4   Stock Flow         86333 non-null  float64       
 5   Stock Consistency  86333 non-null  float64       
 6   Flow Coating       86333 non-null  float64       
 7   Flow Release       86333 non-null  float64       
 8   Jet Wire Ratio     86333 non-null  float64       
 9   Load KWH Refiner   86333 non-null  float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 6.6 MB


In [6]:
raw14.info()

<class 'pandas.DataFrame'>
RangeIndex: 86876 entries, 0 to 86875
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Time               86876 non-null  datetime64[us]
 1   Yankee Speed       86333 non-null  float64       
 2   Pope Reel Speed    86333 non-null  float64       
 3   Yankee Pressure    86333 non-null  float64       
 4   Stock Flow         86333 non-null  float64       
 5   Stock Consistency  86333 non-null  float64       
 6   Flow Coating       86333 non-null  float64       
 7   Flow Release       86333 non-null  float64       
 8   Jet Wire Ratio     86333 non-null  float64       
 9   Load KWH Refiner   86333 non-null  float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 6.6 MB


In [7]:
# Create Categories PM Stop/Run
raw14['PM_stop'] = np.where(raw14['Pope Reel Speed'] == 0, "stop", "run")

# Create Coating/(Area.Min) Feature
raw14['Coating/(Area.Min)'] = ((raw14['Flow Coating'] * 60) / (raw14['Yankee Speed'] * 3050))

In [8]:
raw14.info()

<class 'pandas.DataFrame'>
RangeIndex: 86876 entries, 0 to 86875
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                86876 non-null  datetime64[us]
 1   Yankee Speed        86333 non-null  float64       
 2   Pope Reel Speed     86333 non-null  float64       
 3   Yankee Pressure     86333 non-null  float64       
 4   Stock Flow          86333 non-null  float64       
 5   Stock Consistency   86333 non-null  float64       
 6   Flow Coating        86333 non-null  float64       
 7   Flow Release        86333 non-null  float64       
 8   Jet Wire Ratio      86333 non-null  float64       
 9   Load KWH Refiner    86333 non-null  float64       
 10  PM_stop             86876 non-null  str           
 11  Coating/(Area.Min)  75480 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 8.2 MB


#### Filter Out

In [9]:
# Filter Out Data
raw14 = raw14[(raw14['Yankee Speed'] >= 500) & (raw14['Pope Reel Speed'] >= 400)]

In [10]:
raw14.info()

<class 'pandas.DataFrame'>
Index: 73093 entries, 0 to 86875
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                73093 non-null  datetime64[us]
 1   Yankee Speed        73093 non-null  float64       
 2   Pope Reel Speed     73093 non-null  float64       
 3   Yankee Pressure     73093 non-null  float64       
 4   Stock Flow          73093 non-null  float64       
 5   Stock Consistency   73093 non-null  float64       
 6   Flow Coating        73093 non-null  float64       
 7   Flow Release        73093 non-null  float64       
 8   Jet Wire Ratio      73093 non-null  float64       
 9   Load KWH Refiner    73093 non-null  float64       
 10  PM_stop             73093 non-null  str           
 11  Coating/(Area.Min)  73093 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 7.5 MB


In [11]:
# Delete before-after 0 values in 'Pope Reel Speed'
raw14 = raw14.reset_index(drop=True)
is_zero = np.isclose(raw14['Pope Reel Speed'], 0, atol=1e-5)
mask_to_drop = pd.Series(is_zero).rolling(window=11, center=True, min_periods=1).max().astype(bool)
df_clean = raw14[~mask_to_drop].copy()

In [12]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 73093 entries, 0 to 73092
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                73093 non-null  datetime64[us]
 1   Yankee Speed        73093 non-null  float64       
 2   Pope Reel Speed     73093 non-null  float64       
 3   Yankee Pressure     73093 non-null  float64       
 4   Stock Flow          73093 non-null  float64       
 5   Stock Consistency   73093 non-null  float64       
 6   Flow Coating        73093 non-null  float64       
 7   Flow Release        73093 non-null  float64       
 8   Jet Wire Ratio      73093 non-null  float64       
 9   Load KWH Refiner    73093 non-null  float64       
 10  PM_stop             73093 non-null  str           
 11  Coating/(Area.Min)  73093 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 6.9 MB


In [13]:
df_clean.head()

,Time,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner,PM_stop,Coating/(Area.Min)
0,2026-01-04 00:00:00,1220.076,927.041,7.214,1183.449,3.094,30.273,43.640,0.925,232.963,run,0.000488
1,2026-01-04 00:01:00,1220.076,927.041,7.197,1185.764,3.106,30.275,43.643,0.925,237.371,run,0.000488
2,2026-01-04 00:02:00,1219.761,927.194,7.186,1185.268,3.106,30.271,43.637,0.925,244.352,run,0.000488
3,2026-01-04 00:03:00,1219.761,927.271,7.203,1183.339,3.108,30.267,43.631,0.925,240.107,run,0.000488
4,2026-01-04 00:04:00,1219.918,927.194,7.225,1184.386,3.107,30.271,43.637,0.925,241.781,run,0.000488


## Preparation

### Preparation Process

#### Parameter PM

In [14]:
shift1_start = time(7, 0, 1)
shift1_end   = time(15, 0, 0)
shift2_start = time(15, 0, 1)
shift2_end   = time(23, 0, 0)
def assign_shift(t):
    if shift1_start <= t <= shift1_end:
        return 'Shift 1'
    elif shift2_start <= t <= shift2_end:
        return 'Shift 2'
    else:
        return 'Shift 3'

In [15]:
urutan_params_pm = [
    'Date', 'Time', 'Shift', 'Join_Key', 'Timestamp',
    'Creping', 'Yankee Speed', 'Pope Reel Speed',
    'Yankee Pressure', 'Stock Flow', 'Stock Consistency',
    'Flow Coating', 'Flow Release', 'Jet Wire Ratio',
    'Load KWH Refiner','Key_Date', 'PM_stop', 'Coating/(Area.Min)'
]

In [16]:
def preprocess_pm(df_input):
    df = df_input.copy()
    df = df.drop(columns=['Time'])
    df.columns = df.columns.str.strip()
    df.insert(0, 'Creping', (df['Yankee Speed'] - df['Pope Reel Speed']) * 100 / df['Yankee Speed'])
    def extract_time(x):
        if isinstance(x, str):
            return datetime.strptime(x.split(' ')[1], '%H:%M:%S').time()
        else:  # sudah datetime/Timestamp
            return x.time()
    df.insert(0, 'Time', df_input['Time'].apply(extract_time))
    df['Shift'] = df['Time'].apply(assign_shift)
    
    def extract_date(x):
        if isinstance(x, str):
            return x.split(' ')[0]
        else:
            return x.strftime('%d/%m/%y')
    df.insert(0, 'Date', df_input['Time'].apply(extract_date))
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')
    df['Key_Date'] = [df['Date'][index] - timedelta(days=1) 
                if
                df['Shift'][index] == 'Shift 3' 
                else df['Date'][index] 
                for index in range(len(df['Shift']))]
    df['Join_Key'] = df['Key_Date'].astype(str) + ' ' + df['Time'].astype(str) + ' ' + df['Shift']
    df['Timestamp'] = df['Date'].astype(str) + ' ' + df['Time'].astype(str)
    df['Shift'] = df['Shift'].str.extract(r'(\d+)').astype(int)
    df = df[urutan_params_pm]
    df.drop(columns=['Key_Date'], inplace=True)
    return df

#### Data Reel

In [17]:
def time_to_hms(val):
    s = str(val).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return pd.NA

    # If already contains colon, parse parts directly
    if ':' in s:
        parts = s.split(':')
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 and parts[1] != '' else 0
        sec = int(parts[2]) if len(parts) > 2 and parts[2] != '' else 0

    # If contains dot, treat left as hours and right as minutes (common human shorthand)
    elif '.' in s:
        left, right = s.split('.', 1)
        if int(left) >= 24:
            return pd.NA  # Invalid hour value
        left = 0 if left == "24" else left  # Handle "24" as "00"
        h = int(left) if left != '' else 0

        # If right part is short (1 or 2 digits) treat it as minutes (e.g., "4.1" -> 4:01, "11.55" -> 11:55)
        if len(right) <= 2:
            m = int(right)
            sec = 0
        else:
            # If right part is longer, treat the whole value as a decimal hour (fallback)
            # e.g., "4.125" -> 4.125 hours -> convert fractional hour to minutes
            f = float(s)
            total_minutes = int(round((f - math.floor(f)) * 60))
            m = total_minutes
            sec = 0

    # No separator: treat as hours only (e.g., "6" -> 06:00:00)
    else:
        h = int(float(s))
        m = 0
        sec = 0

    # Normalize minutes >= 60 into hours
    if m >= 60:
        extra_h = m // 60
        h = (h + extra_h) % 24
        m = m % 60

    return f"{h:02d}:{m:02d}:{sec:02d}"

In [18]:
def preprocess_reel(df_input):
    df = df_input.copy()
    df['Time'] = df['Time'].apply(time_to_hms)
    df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%d.%m.%y')
    df = df.dropna(subset = ['Time']).reset_index(drop = True)
    df['Timestamp'] = df['Tanggal'].astype(str) + ' ' + df['Time'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S')
    # Edit Kolom
    df = df.rename(columns={'Tanggal': 'Date'})
    first_cols = ['Date', 'Time', 'Shift', 'Timestamp']
    other_cols = [col for col in df.columns if col not in first_cols]
    df = df[first_cols + other_cols]
    return df

### Apply Preparation

#### Parameter PM

In [19]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 73093 entries, 0 to 73092
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                73093 non-null  datetime64[us]
 1   Yankee Speed        73093 non-null  float64       
 2   Pope Reel Speed     73093 non-null  float64       
 3   Yankee Pressure     73093 non-null  float64       
 4   Stock Flow          73093 non-null  float64       
 5   Stock Consistency   73093 non-null  float64       
 6   Flow Coating        73093 non-null  float64       
 7   Flow Release        73093 non-null  float64       
 8   Jet Wire Ratio      73093 non-null  float64       
 9   Load KWH Refiner    73093 non-null  float64       
 10  PM_stop             73093 non-null  str           
 11  Coating/(Area.Min)  73093 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 6.9 MB


In [20]:
pm_14 = preprocess_pm(df_clean)
pm_14.head()

,Date,Time,Shift,Join_Key,Timestamp,Creping,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner,PM_stop,Coating/(Area.Min)
0,2026-01-04,00:00:00,3,2026-01-03 00:00:00 Shift 3,2026-01-04 00:00:00,24.017766,1220.076,927.041,7.214,1183.449,3.094,30.273,43.640,0.925,232.963,run,0.000488
1,2026-01-04,00:01:00,3,2026-01-03 00:01:00 Shift 3,2026-01-04 00:01:00,24.017766,1220.076,927.041,7.197,1185.764,3.106,30.275,43.643,0.925,237.371,run,0.000488
2,2026-01-04,00:02:00,3,2026-01-03 00:02:00 Shift 3,2026-01-04 00:02:00,23.985600,1219.761,927.194,7.186,1185.268,3.106,30.271,43.637,0.925,244.352,run,0.000488
3,2026-01-04,00:03:00,3,2026-01-03 00:03:00 Shift 3,2026-01-04 00:03:00,23.979288,1219.761,927.271,7.203,1183.339,3.108,30.267,43.631,0.925,240.107,run,0.000488
4,2026-01-04,00:04:00,3,2026-01-03 00:04:00 Shift 3,2026-01-04 00:04:00,23.995383,1219.918,927.194,7.225,1184.386,3.107,30.271,43.637,0.925,241.781,run,0.000488


#### Data Reel

In [21]:
reel_pm14 = preprocess_reel(reel_pm14)
reel_pm14.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-01,07:45:00,1,2026-03-01 07:45:00,FC 12,16,12.36,0.89,413,190,120,32,86.6,Acc Sotiss
1,2026-03-01,08:03:00,1,2026-03-01 08:03:00,FC 12,17,12.68,0.96,306,148,90,30,86.5,Acc Sotiss
2,2026-03-01,09:03:00,1,2026-03-01 09:03:00,FC 12,18,12.50,0.99,352,150,96,32,86.6,Acc Sotiss
3,2026-03-01,10:19:00,1,2026-03-01 10:19:00,FC 12,19,12.24,0.91,365,155,104,32,86.3,Acc Sotiss
4,2026-03-01,11:02:00,1,2026-03-01 11:02:00,FC 12,20,12.34,0.98,294,140,81,32,86.6,Acc Sotiss


## Pipeline

In [22]:
# 1. MEMBACA DATA
df_reel = reel_pm14.copy()
df_params = pm_14.copy()

# 2. KONVERSI TIMESTAMP
df_reel['Timestamp'] = pd.to_datetime(df_reel['Timestamp'])
df_params['Timestamp'] = pd.to_datetime(df_params['Timestamp'])

# 3. SORT KEY
# REEL: Jam 00-06 ditambah 1 hari (karena di Excel tanggalnya mundur 1 hari dari params)
def create_sort_key_reel(ts):
    if ts.hour < 7:
        return ts + timedelta(days=1)
    return ts

df_reel['Sort_Key'] = df_reel['Timestamp'].apply(create_sort_key_reel)
df_params['Sort_Key'] = df_params['Timestamp']  # Params tidak perlu adjustment

# 4. FILTER BERDASARKAN SORT_KEY RANGE PARAMS
params_sort_min = df_params['Sort_Key'].min()
params_sort_max = df_params['Sort_Key'].max()

df_reel_filtered = df_reel[
    (df_reel['Sort_Key'] >= params_sort_min) & 
    (df_reel['Sort_Key'] <= params_sort_max)
].copy()

df_reel_filtered = df_reel_filtered.sort_values('Sort_Key').reset_index(drop=True)

# 5. DAFTAR VARIABEL
cols_to_avg = [
    'Creping', 'Yankee Speed', 'Pope Reel Speed', 'Yankee Pressure',
    'Stock Flow', 'Stock Consistency', 'Flow Coating', 'Flow Release',
    'Jet Wire Ratio', 'Load KWH Refiner', 'Coating/(Area.Min)'
]

# 6. LOOPING GROUPBY AVERAGE
results = []

for i in range(len(df_reel_filtered) - 1):
    start_time = df_reel_filtered['Timestamp'].iloc[i]
    end_time = df_reel_filtered['Timestamp'].iloc[i + 1]
    start_sort = df_reel_filtered['Sort_Key'].iloc[i]
    end_sort = df_reel_filtered['Sort_Key'].iloc[i + 1]
    
    mask = (df_params['Sort_Key'] >= start_sort) & (df_params['Sort_Key'] < end_sort)
    df_filtered = df_params.loc[mask]
    
    if len(df_filtered) == 0:
        continue
    
    row = {
        'Start_Time': start_time,
        'End_Time': end_time,
        'Data_Count': len(df_filtered)
    }
    
    for col in cols_to_avg:
        mean_val = df_filtered[col].mean()
        row[f'Mean_{col}'] = round(mean_val, 6) if pd.notna(mean_val) else None
    
    results.append(row)

# 7. HASIL
df_result = pd.DataFrame(results)

# 8. SIMPAN
# df_result.to_excel('grouby_params.xlsx', index=False)
print("\n✅ File disimpan: grouby_params.xlsx")


✅ File disimpan: grouby_params.xlsx


# Join Table

## Joining Df_Results and Reel Data

In [23]:
df_groupby = df_result.copy()
df_groupby.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,Mean_Flow Release,Mean_Jet Wire Ratio,Mean_Load KWH Refiner,Mean_Coating/(Area.Min)
0,2026-03-03 23:03:00,2026-03-03 00:02:00,2,24.007426,1230.084000,934.772500,7.208500,1151.000500,3.103000,30.524000,44.002000,0.925,237.140000,0.000488
1,2026-03-03 00:02:00,2026-03-03 02:05:00,123,23.903351,1229.941382,935.944171,7.199691,1150.041789,3.099203,30.519309,43.995447,0.925,236.729309,0.000488
2,2026-03-03 02:05:00,2026-03-03 03:22:00,77,23.865762,1229.948130,936.411649,7.199740,1149.928558,3.099727,30.519558,43.995792,0.925,236.195260,0.000488
3,2026-03-03 03:22:00,2026-03-03 04:17:00,55,23.888775,1228.324709,934.894782,7.199782,1151.660382,3.099345,30.479382,43.937800,0.925,240.392491,0.000488
4,2026-03-03 04:17:00,2026-03-03 05:12:00,55,23.988053,1219.965582,927.319527,7.200109,1159.977818,3.100491,30.272164,43.638709,0.925,238.305873,0.000488


In [24]:
df_reel = reel_pm14.copy()
df_reel.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-01,07:45:00,1,2026-03-01 07:45:00,FC 12,16,12.36,0.89,413,190,120,32,86.6,Acc Sotiss
1,2026-03-01,08:03:00,1,2026-03-01 08:03:00,FC 12,17,12.68,0.96,306,148,90,30,86.5,Acc Sotiss
2,2026-03-01,09:03:00,1,2026-03-01 09:03:00,FC 12,18,12.50,0.99,352,150,96,32,86.6,Acc Sotiss
3,2026-03-01,10:19:00,1,2026-03-01 10:19:00,FC 12,19,12.24,0.91,365,155,104,32,86.3,Acc Sotiss
4,2026-03-01,11:02:00,1,2026-03-01 11:02:00,FC 12,20,12.34,0.98,294,140,81,32,86.6,Acc Sotiss


In [25]:
# Create Join Key in df_groupby
df_groupby['Join_Key_Timestamp'] = df_groupby['End_Time'] #End_Time
# Create Join Key in df_reel
df_reel['Join_Key_Timestamp'] = df_reel['Timestamp']

In [26]:
# Join df_groupby with df_reel on Join_Key_Timestamp
df_joined = pd.merge(df_groupby, df_reel, left_on='Join_Key_Timestamp', right_on='Join_Key_Timestamp', how='inner')
df_joined.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,...,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-03 23:03:00,2026-03-03 00:02:00,2,24.007426,1230.084000,934.772500,7.208500,1151.000500,3.103000,30.524000,...,FC 16,84,16.14,1.11,298,117,84,32,87.6,Acc Sotiss
1,2026-03-03 00:02:00,2026-03-03 02:05:00,123,23.903351,1229.941382,935.944171,7.199691,1150.041789,3.099203,30.519309,...,FC 16,85,15.72,1.04,345,140,97,31,87.5,Acc Sotiss
2,2026-03-03 02:05:00,2026-03-03 03:22:00,77,23.865762,1229.948130,936.411649,7.199740,1149.928558,3.099727,30.519558,...,FC 16,86,16.00,1.05,315,126,92,31,87.8,Acc Sotiss
3,2026-03-03 03:22:00,2026-03-03 04:17:00,55,23.888775,1228.324709,934.894782,7.199782,1151.660382,3.099345,30.479382,...,FC 16,87,15.94,1.07,281,118,79,28,87.1,Acc Sotiss
4,2026-03-03 04:17:00,2026-03-03 05:12:00,55,23.988053,1219.965582,927.319527,7.200109,1159.977818,3.100491,30.272164,...,FC 16,88,16.20,1.09,249,109,73,29,88.3,Acc Sotiss


## Joining df_joined with BB Table

### Data BB

In [27]:
# Pipeline for Data Reel
def merge_BB_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file, engine='openpyxl')
            dataframes.append(df)
            df.drop(columns=['Grade'], inplace=True)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [28]:
# Eksekusi Pipeline
file_sources = [
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM14.xlsx",
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM14.xlsx"
]
df_BB = merge_BB_data(file_sources)
df_BB.tail()

Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM14.xlsx | Shape: (31, 8)
Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM14.xlsx | Shape: (31, 8)


<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_3212\549467096.py:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
  "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM14.xlsx",
C:\Users\user\AppData\Local\Temp\ipykernel_3212\549467096.py:4: SyntaxWarning: "\K" is an invalid escape sequence. 

,Date,GSM,Total NBKP,Total LBKP,Sub Total Pulp+Broke,% NBKP,% LBKP,% Pulp+Broke
57,2026-04-27,"15/13,5/13",5812.5,0,47064.076,12.350184,0,1130843.560
58,2026-04-28,"13/12,5",6187.5,0,42408.396,14.590271,0,1173251.956
59,2026-04-29,"12,5",6562.5,0,34734.308,18.893424,0,1207986.264
60,2026-04-30,"12,5/12",5625.0,0,33796.808,16.643584,0,1241783.072
61,2026-05-01,0,0.0,0,0.000,0.000000,0,1241783.072


df_BB = pd.read_excel("../Efficiency/BB_Maret_PM14.xlsx", engine='openpyxl')
df_BB.drop(columns=['Grade'], inplace=True)
df_BB.head()

### Last Join

In [29]:
# Standarisasi kolom "date" ke bentuk datetime
df_joined['Date'] = pd.to_datetime(df_joined['Date'])
df_BB['Date'] = pd.to_datetime(df_BB['Date'])

In [30]:
# Joining
df_final = pd.merge(df_joined, df_BB, on='Date', how='inner') #inner
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Start_Time               906 non-null    datetime64[us]
 1   End_Time                 906 non-null    datetime64[us]
 2   Data_Count               906 non-null    int64         
 3   Mean_Creping             906 non-null    float64       
 4   Mean_Yankee Speed        906 non-null    float64       
 5   Mean_Pope Reel Speed     906 non-null    float64       
 6   Mean_Yankee Pressure     906 non-null    float64       
 7   Mean_Stock Flow          906 non-null    float64       
 8   Mean_Stock Consistency   906 non-null    float64       
 9   Mean_Flow Coating        906 non-null    float64       
 10  Mean_Flow Release        906 non-null    float64       
 11  Mean_Jet Wire Ratio      906 non-null    float64       
 12  Mean_Load KWH Refiner    906 non-null    float6

In [31]:
# Membaca file sebelumnya
df = df_final.copy()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[~df['GSM'].str.contains('/', na=False)].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)
# Cleaning - MDWT
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna(subset=['MDWT']).copy()

In [32]:
df['GSM'].tail(50)

691    14.5
692    14.5
693    14.5
694    14.5
695    14.5
696    14.5
697    14.5
698    14.5
699    14.5
700    14.5
701    14.5
702    14.5
703    14.5
704    14.5
705    14.5
706    14.5
707    14.5
708    14.5
709    14.5
710    14.5
711    14.5
712    14.5
713    14.5
714    14.5
715    14.5
716    14.5
717    14.5
718    14.5
719    14.5
720    14.5
721    14.5
865    12.5
866    12.5
867    12.5
869    12.5
870    12.5
871    12.5
872    12.5
873    12.5
874    12.5
875    12.5
876    12.5
877    12.5
878    12.5
879    12.5
880    12.5
881    12.5
882    12.5
883    12.5
884    12.5
Name: GSM, dtype: float64

### Convert Final Data to Excel

#### Group by Grade

In [33]:
# Memisahkan data berdasarkan awalan pada kolom Grade
df_toilet = df[df['Grade'].str.startswith('T', na=False) & 
               ~df['Grade'].str.startswith('TW', na=False)]
df_towel = df[df['Grade'].str.startswith('TW', na=False)]
df_facial = df[df['Grade'].str.startswith('FC', na=False)]

# Menampilkan jumlah data
print("Jumlah data Toilet :", len(df_toilet))
print("Jumlah data Towel  :", len(df_towel))
print("Jumlah data Facial :", len(df_facial))

Jumlah data Toilet : 233
Jumlah data Towel  : 0
Jumlah data Facial : 177


Simpan ke file Excel terpisah

In [34]:
df.to_excel('Final_PM14.xlsx', index=False)

In [35]:
df_toilet.to_excel("Final_PM14-Toilet.xlsx", index=False)
#df_towel.to_excel("Final_PM14-Towel.xlsx", index=False)
df_facial.to_excel("Final_PM14-Facial.xlsx", index=False)